**Data Info**

train.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용
* first_party_winner : 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

test.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용

sample_submission.csv - 제출 양식
* ID : 사건 샘플 ID
* first_party_winner : 예측한 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

# Fine-tune Model: Legal-BERT
법률 도메인 특화 모델 Legal-BERT 이용

In [ ]:
!pip install -U transformers -q
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from transformers import EarlyStoppingCallback

In [ ]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [ ]:
# Data Preprocessing
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train['facts'].tolist(),
    train['first_party_winner'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train['first_party_winner']
)

test_texts = test['facts'].tolist()

In [ ]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# mapping
class CaseDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = CaseDataset(train_texts, train_labels)
val_dataset = CaseDataset(val_texts, val_labels)
test_dataset = CaseDataset(test_texts)

In [ ]:
class_counts = train['first_party_winner'].value_counts().sort_index()
class_weights = torch.tensor( #클래스별 가중치 계산
    [len(train) / (2 * c) for c in class_counts],
    dtype=torch.float
).to('cuda')

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Fine-tune Model: Legal-BERT

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir='./bert_result',
    num_train_epochs=6, # early stopping 추가
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=1e-5, #과적합으로 2e-5에서 내림
    warmup_steps=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,  # GPU 지원 시 학습 속도 향상
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
1,0.709554,0.703404,0.397177,0.246851,0.372165
2,0.696817,0.707113,0.467742,0.456790,0.467525
3,0.682423,0.725747,0.649194,0.770449,0.513429
4,0.624480,0.744766,0.600806,0.693498,0.560622
5,0.546197,0.788802,0.612903,0.720930,0.544676
6,0.504196,0.799512,0.600806,0.695385,0.558219


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1488, training_loss=0.6267077704911591, metrics={'train_runtime': 521.2469, 'train_samples_per_second': 22.815, 'train_steps_per_second': 2.855, 'total_flos': 3128916670341120.0, 'train_loss': 0.6267077704911591, 'epoch': 6.0})

In [ ]:
# Submission
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)

submit = pd.read_csv('./sample_submission.csv')
submit['first_party_winner'] = pred_labels
submit.to_csv('./bert_submit.csv', index=False)
print('Done')

Done
